# Prepare SCI data
In this notebook, we select the necessary emission-related variables from the Scenario Compass Database

In [1]:
import pyam
import pandas as pd

import dotenv
import os
from pathlib import Path

import datetime

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
dotenv.load_dotenv()

True

In [3]:
df = pyam.IamDataFrame(
    Path(
        os.environ['SCI_DATA']
    )
)

[INFO] 10:13:29 - pyam.core: Reading file /Users/ganti/Documents/data/sci-data/SCI-2025_v1.0_pathways_ensemble_global.xlsx


We have a lot of variables available but only need relevant emission-related variables for this analysis.
For this, we downselect the variables that are present in the historical emissions data that is used for CMIP7: https://zenodo.org/records/17845154

In [4]:
hist_data = pyam.IamDataFrame(
    Path(
        os.environ['HIST_DATA']
    )
)

[INFO] 10:13:54 - pyam.core: Reading file /Users/ganti/Documents/data/cmip7-data/global-workflow-history_202511261223_202511040855_202512032146_202512021030_7e32405ade790677a6022ff498395bff00d9792d_202511040855_202512071232_202511040855_202511040855_0002_0002.csv


We check for the intersection of the two sets of variables.

In [5]:
variable_intersection = list(
    set(df.variable).intersection(hist_data.variable)
)

What is present?

In [6]:
variable_intersection, len(variable_intersection), len(hist_data.variable)

(['Emissions|C2F6',
  'Emissions|HFC|HFC23',
  'Emissions|OC',
  'Emissions|CO2|AFOLU',
  'Emissions|NH3',
  'Emissions|N2O',
  'Emissions|SF6',
  'Emissions|HFC|HFC32',
  'Emissions|HFC|HFC227ea',
  'Emissions|CO',
  'Emissions|VOC',
  'Emissions|NOx',
  'Emissions|BC',
  'Emissions|CF4',
  'Emissions|CH4',
  'Emissions|HFC|HFC125',
  'Emissions|CO2|Energy and Industrial Processes',
  'Emissions|Sulfur',
  'Emissions|HFC|HFC143a',
  'Emissions|HFC|HFC245fa',
  'Emissions|HFC|HFC134a',
  'Emissions|C6F14',
  'Emissions|HFC|HFC43-10'],
 23,
 52)

What is missing?

In [7]:
[x for x in hist_data.variable if x not in variable_intersection]

['Emissions|C3F8',
 'Emissions|C4F10',
 'Emissions|C5F12',
 'Emissions|C7F16',
 'Emissions|C8F18',
 'Emissions|CCl4',
 'Emissions|CFC11',
 'Emissions|CFC113',
 'Emissions|CFC114',
 'Emissions|CFC115',
 'Emissions|CFC12',
 'Emissions|CH2Cl2',
 'Emissions|CH3Br',
 'Emissions|CH3CCl3',
 'Emissions|CH3Cl',
 'Emissions|CHCl3',
 'Emissions|HCFC141b',
 'Emissions|HCFC142b',
 'Emissions|HCFC22',
 'Emissions|HFC|HFC152a',
 'Emissions|HFC|HFC236fa',
 'Emissions|HFC|HFC365mfc',
 'Emissions|Halon1202',
 'Emissions|Halon1211',
 'Emissions|Halon1301',
 'Emissions|Halon2402',
 'Emissions|NF3',
 'Emissions|SO2F2',
 'Emissions|cC4F8']

Seems fine so far. These will be handled later..

In [8]:
df_downselected = (
    df
    .filter(
        variable=variable_intersection
    )
)

In [9]:
df_downselected.require_data(
    year=2100,
    exclude_on_fail=True
)

[INFO] 10:13:54 - pyam.validation: 22 scenarios failed validation and will be set as `exclude=True`.


,model,scenario,year
0,IMACLIM 2.0,NAVIGATE Structural Change-Baseline,2100
1,IMACLIM 2.0,NAVIGATE Structural Change-Net-Zero,2100
2,PROMETHEUS 1.2,ECEMF-DIAG-C400-lin,2100
3,PROMETHEUS 1.2,ECEMF-DIAG-NPi,2100
4,PROMETHEUS V1,IAM-COMPACT-Current-Policies-Emissions-Intensity,2100
5,PROMETHEUS V1,IAM-COMPACT-NDC-Emissions-Intensity,2100
6,PROMETHEUS V1,IAM-COMPACT-NDC-Long-Term-Pledges,2100
7,PROMETHEUS V1,NAVIGATE Demand-1.5°C-act_u,2100
8,PROMETHEUS V1,NAVIGATE Demand-1.5°C-all_u,2100
9,PROMETHEUS V1,NAVIGATE Demand-1.5°C-ele_u,2100


In [10]:
df_downselected.filter(
    exclude=False,
    inplace=True
)

Let us interpolate for missing years and save this out for further analysis

In [11]:
df_downselected.interpolate(
    time=range(2010, 2101),
    inplace=True
)

In [12]:
df_downselected.to_csv(
    Path(
        f'output/100_inputdata_{datetime.date.today().strftime("%Y%m%d")}.csv'.format()
    )
)